# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant JSON-LD schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Date Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets and their fields/columns using their `@id` values.

In [ ]:
# Display all available RecordSets, their @id, and their fields' @id
def overview_recordsets(ds):
    print("Available RecordSets and Fields:\n")
    for record_set in ds.record_sets:
        rset_id = record_set['@id']
        name = record_set.get('name', '(no name)')
        print(f"RecordSet: {name}\n  @id: {rset_id}")
        fields = record_set.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        if fields:
            print("  Fields:")
            for field in fields:
                if isinstance(field, dict):
                    field_id = field.get('@id', field)
                else:
                    field_id = field
                print(f"    - {field_id}")
        else:
            print("  (No fields found)")
        print()

# Print the overview (record sets and their fields by @id)
overview_recordsets(dataset)

## 3. Data Extraction
Let's extract tabular data from each record set defined in the dataset. We use the record set and field `@id`s as determined in the previous section.

> **Note**: Below, you'll want to consult the RecordSets as printed above and select their `@id`s for analysis. We'll demonstrate loading the first discovered RecordSet for further analysis.

In [ ]:
# List all RecordSet @ids
record_sets = [rs['@id'] for rs in dataset.record_sets]
print("All RecordSet @ids:", record_sets)

# Extract a sample record for each RecordSet
sampled = {}
for rs_id in record_sets:
    print(f"\nFirst 2 records from {rs_id}:")
    records_iter = dataset.records(record_set=rs_id)
    for i, rec in enumerate(records_iter):
        print(rec)
        if i >= 1:
            break

Now, let's load all records for a chosen RecordSet into a DataFrame. For the purposes of demonstration, we'll use the first RecordSet, but you can substitute any other valid RecordSet `@id` from your dataset.

In [ ]:
# Choose a RecordSet for DataFrame analysis
# If you know a specific record set @id, set record_set_id = '<@id>'
if len(record_sets) == 0:
    raise Exception("No RecordSets found in this dataset.")
record_set_id = record_sets[0]

# Load all records from the chosen RecordSet into a DataFrame
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)

print(f"Fields (@id) in DataFrame for RecordSet {record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Now, let's analyze, filter, and process data. We'll:
- Select a numeric field (by its `@id`)
- Filter rows where the numeric field exceeds a threshold value
- Normalize the numeric column
- Optionally, group by a categorical field

**Substitute the field @id variables in the next cell for your dataset!**

In [ ]:
# List all fields to help identify numeric fields
print("Available fields:")
print(df.columns.tolist())

# Choose a numeric field by its @id (update this if needed)
# Example: 'cr:abundance' or an appropriate numeric field from previous cell's output
numeric_field_id = None
for col in df.columns:
    # Try a simple heuristic: if column has numeric type data, use it
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is None:
    print("No numeric fields automatically detected. Please set `numeric_field_id` manually to a correct @id.")
else:
    print(f"Using numeric field: {numeric_field_id}")
    threshold = df[numeric_field_id].quantile(0.9)  # For demo: top 10% (modify as appropriate)
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Try grouping by a categorical field if available
    group_field = None
    for col in df.columns:
        if col != numeric_field_id and df[col].nunique() < (0.1 * len(df)):
            group_field = col
            break

    if group_field and group_field in filtered_df.columns:
        grouped = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"\nGrouped mean {numeric_field_id} by {group_field}:")
        print(grouped.head())

## 5. Visualization
Let's visualize the distribution of our selected numeric field and, if available, relationships with a categorical field.

Run the cells below to generate common visualizations. Feel free to use other fields for advanced plots!

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field_id and numeric_field_id in df.columns:
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    # If a grouping field is available, boxplot
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field selected for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR<sup>2</sup> mass spectrometry dataset using `mlcroissant`. We:
- Loaded metadata and explored the schema
- Extracted tabular data from the first available RecordSet
- Performed basic EDA, normalization, and grouping
- Visualized data distributions

For deeper analysis, refer to specific @id fields and explore other available RecordSets and columns. See [mlcroissant documentation](https://mlcommons.org/croissant/) for more advanced usage!